# LLM-Based Narrative Block Extraction for DMPBridge

## Objective

The goal of this experiment is to evaluate whether a Large Language Model (Llama 3.1 8B via Ollama) can accurately identify and extract the narrative structure of Data Management Plans (DMPs).

Rather than generating a final DMP JSON representation, the LLM is used only to classify and label portions of DMP text into narrative blocks. The focus of this phase is structure extraction and content preservation.

---

## Workflow

```text
PDF
↓
PDFPlumber Extraction
↓
Markdown Text
↓
Llama 3.1 8B (Ollama)
↓
Structured Blocks
```

At this stage, the workflow stops after structured block generation. Narrative JSON construction and schema mapping will be evaluated separately.

---

## Input

The input is a markdown version of a Data Management Plan generated from the PDF extraction pipeline.

The markdown contains the textual content extracted from the original DMP PDF and serves as the source text for LLM-based structure extraction.

---

## Structured Block Extraction

The LLM analyzes the DMP text and labels each portion of the document using one of four narrative block types.

### Supported Labels

| Label          | Description                                             |
| -------------- | ------------------------------------------------------- |
| document_title | Main title of the DMP                                   |
| section        | Major DMP heading                                       |
| subsection     | Explicit prompt, question, or subheading                |
| content        | Narrative body text, instructions, guidance, or answers |

### Extraction Philosophy

The LLM is instructed to perform extraction only.

The model should:

* Preserve original wording.
* Preserve original ordering.
* Detect existing document structure.
* Avoid creating new headings.
* Avoid summarizing content into headings.
* Avoid hallucinating section or subsection titles.
* Label uncertain text as content rather than inventing structure.

The objective is to identify structure that already exists within the DMP rather than generating new structure.

---

## Example Structured Blocks

```json
[
  {
    "label": "document_title",
    "text": "DATA MANAGEMENT AND SHARING PLAN"
  },
  {
    "label": "section",
    "text": "Element 1: Data Type:"
  },
  {
    "label": "subsection",
    "text": "A. Types and amount of scientific data expected to be generated in the project:"
  },
  {
    "label": "content",
    "text": "This project will generate..."
  }
]
```

---

## Evaluation Criteria

### Structure Detection

Evaluate whether the LLM correctly identifies:

* Document titles
* Major section headings
* Subsection or prompt headings
* Narrative content

### Content Preservation

Evaluate whether:

* Narrative text is preserved
* Content ordering is maintained
* No content is lost
* No content is hallucinated
* Original wording is retained

### Extraction Quality

Evaluate whether:

* Existing headings are detected correctly
* Sentences are not incorrectly converted into headings
* Paragraphs remain content
* The LLM avoids inventing structure

---

## Expected Output

```json
[
  {
    "label": "section",
    "text": "Types of Data"
  },
  {
    "label": "content",
    "text": "The project will generate..."
  }
]
```

The output should represent the structure already present in the DMP while preserving the original text.

---

## Current Research Questions

1. Can an LLM identify DMP structure more accurately than rule-based approaches?
2. Which DMP formats are easiest or most difficult for the LLM to structure?
3. How well does the LLM generalize across different funder and institutional DMP templates?
4. Does the LLM preserve narrative content while extracting structure?
5. What types of structural errors occur most frequently?

---

## Future Work

1. Compare LLM-based block extraction with rule-based extraction methods.
2. Evaluate section and subsection detection accuracy across diverse DMP formats.
3. Measure content preservation using:

   * Word Capture
   * ROUGE-L
   * Word Precision
   * Word Recall
   * Word F1
4. Convert extracted blocks into DMPTool-compatible narrative JSON.
5. Extend extraction to support the RDA Common Standard and DMPTool extension schema.
6. Compare multiple open-source models, including Llama, Qwen, and Mistral, for DMP narrative structure extraction.


In [ ]:
from pathlib import Path

from dmpbridge.llm.llama_client import load_llama
from dmpbridge.llm.llm_narrative_blocks import (
    generate_structured_blocks_with_llm,
    save_blocks,
)



# Project root


cwd = Path.cwd()

if (cwd / "data").exists() and (cwd / "src").exists():
    project_root = cwd
else:
    project_root = cwd.parent

print("Project root:", project_root)



# Input / Output directories


markdown_dir = (
    project_root
    / "data"
    / "pdfplumber_extracted_markdown"
)

blocks_output_dir = (
    project_root
    / "data"
    / "llama_structured_blocks"
)

blocks_output_dir.mkdir(
    parents=True,
    exist_ok=True
)



# Load Llama once


llm = load_llama(
    model_name="llama3.1:8b",
    temperature=0,
)

print("Llama loaded successfully.")



# Process all markdown files


markdown_files = sorted(
    markdown_dir.glob("*.md")
)

print(f"\nFound {len(markdown_files)} markdown files")

for markdown_path in markdown_files:

    sample_name = markdown_path.stem

    print("\n" + "=" * 80)
    print(f"Processing: {sample_name}")
    print("=" * 80)

    try:
       
        # Load markdown text
     

        dmp_text = markdown_path.read_text(
            encoding="utf-8"
        )

        
        # Generate structured blocks only
       

        structured_blocks = generate_structured_blocks_with_llm(
            llm=llm,
            dmp_text=dmp_text,
        )

        
        # Save structured blocks only
        

        blocks_output_path = (
            blocks_output_dir
            / f"{sample_name}_llama_blocks.json"
        )

        save_blocks(
            blocks=structured_blocks,
            output_path=blocks_output_path,
        )

        print(
            f"Saved {len(structured_blocks)} structured blocks: "
            f"{blocks_output_path.name}"
        )

    except Exception as e:
        print(f"ERROR processing {sample_name}")
        print(e)

print("\nFinished processing all markdown files.")

Project root: c:\Users\Nahid\dmpbridge
Llama loaded successfully.

Found 10 markdown files

Processing: sample1
Saved 35 structured blocks: sample1_llama_blocks.json

Processing: sample10
Saved 24 structured blocks: sample10_llama_blocks.json

Processing: sample2
Saved 21 structured blocks: sample2_llama_blocks.json

Processing: sample3
Saved 12 structured blocks: sample3_llama_blocks.json

Processing: sample4
Saved 43 structured blocks: sample4_llama_blocks.json

Processing: sample5
Saved 43 structured blocks: sample5_llama_blocks.json

Processing: sample6
Saved 11 structured blocks: sample6_llama_blocks.json

Processing: sample7
Saved 9 structured blocks: sample7_llama_blocks.json

Processing: sample8
Saved 25 structured blocks: sample8_llama_blocks.json

Processing: sample9
Saved 23 structured blocks: sample9_llama_blocks.json

Finished processing all markdown files.
